In [ ]:
# ========== 1. IMPORT THƯ VIỆN ========== #
import os
import numpy as np
import tensorflow as tf   # <- tf phải được tạo trước
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ========== GPU CONFIG (KAGGLE) ========== #
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU is ready ✅")
    except RuntimeError as e:
        print(e)

# confirm mixed precision
print("Compute dtype:", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)

In [ ]:
## MobileNetV2 21-06 - MULTI-RUN EXPERIMENT
# ========== 1. IMPORT THU VIEN ========== #
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as model_preprocess
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix

# ========== 2. MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")

# ========== 3. DATA PATH & CONFIGURATION ========== #
DATA_DIR = "/kaggle/input/datasets/usertesttttt1/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = "/kaggle/working/report_MobileNetV2_MultiRun"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

MODEL_FILENAME = "mobilenetv2_best.keras"
RANDOM_SEEDS = [42, 123, 456]
print(f"Running {len(RANDOM_SEEDS)} experiments with seeds: {RANDOM_SEEDS}")

all_runs_results = []

# ========== 4. FUNCTION DEFINITIONS ========== #
def create_generators(data_dir, input_size, batch_size=32, seed=None):
    train_datagen = ImageDataGenerator(
        preprocessing_function=model_preprocess,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.7, 1.3],
        fill_mode='nearest'
    )
    val_datagen = ImageDataGenerator(preprocessing_function=model_preprocess)

    train_gen = train_datagen.flow_from_directory(
        os.path.join(data_dir, 'train'),
        target_size=input_size,
        batch_size=batch_size,
        class_mode='categorical',
        seed=seed
    )
    val_gen = val_datagen.flow_from_directory(
        os.path.join(data_dir, 'val'),
        target_size=input_size,
        batch_size=batch_size,
        class_mode='categorical',
        seed=seed
    )
    test_gen = val_datagen.flow_from_directory(
        os.path.join(data_dir, 'test'),
        target_size=input_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )
    return train_gen, val_gen, test_gen

# ========== 5. MAIN EXPERIMENT LOOP ========== #
input_shape = (224, 224, 3)
batch_size = 32
epochs = 30

for run_idx, seed in enumerate(RANDOM_SEEDS):
    print("\n" + "="*80)
    print(f"STARTING RUN {run_idx + 1}/{len(RANDOM_SEEDS)} - SEED: {seed}")
    print("="*80 + "\n")

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_generator, val_generator, test_generator = create_generators(
        DATA_DIR, input_shape[:2], batch_size, seed=seed
    )

    class_weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_generator.classes),
        y=train_generator.classes
    )
    class_weights = dict(enumerate(class_weights))

    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-5))(x)
    x = Dropout(0.5)(x)
    outputs = Dense(len(train_generator.class_indices), activation='softmax', dtype='float32')(x)

    model = Model(inputs=base_model.input, outputs=outputs)

    steps_per_epoch = train_generator.samples // batch_size
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=1e-5,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True
    )
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

    model.compile(
        optimizer=optimizer,
        loss=CategoricalCrossentropy(label_smoothing=0.15),
        metrics=['accuracy']
    )

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, MODEL_FILENAME),
                        save_best_only=True, monitor='val_loss', verbose=1)
    ]

    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs,
        class_weight=class_weights,
        callbacks=callbacks
    )

    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title(f'Accuracy - Run {run_idx+1} (Seed {seed})')
    plt.legend()
    plt.grid()
    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'Loss - Run {run_idx+1} (Seed {seed})')
    plt.legend()
    plt.grid()
    plt.savefig(os.path.join(RESULT_DIR, "learning_curve.png"), dpi=300)
    plt.close()

    model = load_model(os.path.join(RESULT_DIR, MODEL_FILENAME))
    test_generator.reset()

    pred_probs = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(pred_probs, axis=1)
    y_true = test_generator.classes
    class_names = list(test_generator.class_indices.keys())

    report = classification_report(
        y_true, y_pred, target_names=class_names,
        output_dict=True, digits=4, zero_division=0
    )

    report_text = classification_report(
        y_true, y_pred, target_names=class_names,
        digits=4, zero_division=0
    )
    with open(os.path.join(RESULT_DIR, "classification_report.txt"), "w") as f:
        f.write(report_text)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7,6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names,
                yticklabels=class_names,
                cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - Run {run_idx+1} (Seed {seed})")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "confusion_matrix.png"), dpi=300)
    plt.close()

    test_acc = np.mean(y_pred == y_true)
    overall_precision = report['weighted avg']['precision']
    overall_recall = report['weighted avg']['recall']
    overall_f1 = report['weighted avg']['f1-score']

    per_class_metrics = {}
    for class_name in class_names:
        per_class_metrics[class_name] = {
            'precision': report[class_name]['precision'],
            'recall': report[class_name]['recall'],
            'f1-score': report[class_name]['f1-score']
        }

    run_results = {
        'run': run_idx + 1,
        'seed': seed,
        'accuracy': test_acc,
        'precision': overall_precision,
        'recall': overall_recall,
        'f1_score': overall_f1,
        'macro_precision': report['macro avg']['precision'],
        'macro_recall': report['macro avg']['recall'],
        'macro_f1_score': report['macro avg']['f1-score'],
        'per_class_metrics': per_class_metrics,
        'result_dir': RESULT_DIR,
        'history': history.history,
        'y_true': y_true,
        'y_pred': y_pred,
        'pred_probs': pred_probs,
        'class_names': class_names,
        'test_filenames': test_generator.filenames,
        'n_train': train_generator.samples,
        'n_val': val_generator.samples,
        'n_test': test_generator.samples
    }

    all_runs_results.append(run_results)

    print(f"\nRUN {run_idx + 1} COMPLETED")
    print(f"   Accuracy: {test_acc:.4f}")
    print(f"   Precision (weighted): {overall_precision:.4f}")
    print(f"   Recall (weighted): {overall_recall:.4f}")
    print(f"   F1-Score (weighted): {overall_f1:.4f}")

    tf.keras.backend.clear_session()

print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETED!")
print("="*80)


In [ ]:
# ========== AGGREGATE RESULTS FROM ALL RUNS ========== #
print("\n" + "="*80)
print("📊 AGGREGATING RESULTS FROM ALL RUNS")
print("="*80 + "\n")

# Extract overall metrics
accuracies = [r['accuracy'] for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls = [r['recall'] for r in all_runs_results]
f1_scores = [r['f1_score'] for r in all_runs_results]

# Calculate mean and std
overall_stats = {
    'Accuracy': {
        'mean': np.mean(accuracies),
        'std': np.std(accuracies),
        'values': accuracies
    },
    'Precision': {
        'mean': np.mean(precisions),
        'std': np.std(precisions),
        'values': precisions
    },
    'Recall': {
        'mean': np.mean(recalls),
        'std': np.std(recalls),
        'values': recalls
    },
    'F1-Score': {
        'mean': np.mean(f1_scores),
        'std': np.std(f1_scores),
        'values': f1_scores
    }
}

# Print summary
print("OVERALL METRICS ACROSS ALL RUNS:")
print("-" * 80)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} ± {stats['std']:.4f}")
    print(f"              Individual runs: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 80)

# Get class names from first run
class_names = list(all_runs_results[0]['per_class_metrics'].keys())

# Calculate per-class statistics
per_class_stats = {}
for class_name in class_names:
    per_class_stats[class_name] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][class_name][metric] for r in all_runs_results]
        per_class_stats[class_name][metric] = {
            'mean': np.mean(values),
            'std': np.std(values),
            'values': values
        }

print("\n✅ Statistics calculated successfully!")

In [ ]:
# ========== CREATE SCIENTIFIC REPORT TABLE ========== #
print("\n" + "="*80)
print("📋 CREATING SCIENTIFIC REPORT TABLES")
print("="*80 + "\n")

# Create overall metrics table
overall_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Mean': [overall_stats['Accuracy']['mean'],
             overall_stats['Precision']['mean'],
             overall_stats['Recall']['mean'],
             overall_stats['F1-Score']['mean']],
    'Std': [overall_stats['Accuracy']['std'],
            overall_stats['Precision']['std'],
            overall_stats['Recall']['std'],
            overall_stats['F1-Score']['std']],
    'Run 1': [accuracies[0], precisions[0], recalls[0], f1_scores[0]],
    'Run 2': [accuracies[1], precisions[1], recalls[1], f1_scores[1]],
    'Run 3': [accuracies[2], precisions[2], recalls[2], f1_scores[2]]
})

# Format for scientific presentation
overall_df['Mean ± Std'] = overall_df.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['Std']:.4f}", axis=1
)

print("\n📊 OVERALL PERFORMANCE METRICS (3 RUNS)")
print("="*80)
print(overall_df[['Metric', 'Mean ± Std', 'Run 1', 'Run 2', 'Run 3']].to_string(index=False))
print("="*80)

# Save to CSV
overall_df.to_csv(os.path.join(BASE_RESULT_DIR, "overall_metrics_summary.csv"), index=False)

# Create per-class metrics table
per_class_rows = []
for class_name in class_names:
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        per_class_rows.append({
            'Class': class_name,
            'Metric': metric.capitalize(),
            'Mean': stats['mean'],
            'Std': stats['std'],
            'Mean ± Std': f"{stats['mean']:.4f} ± {stats['std']:.4f}",
            'Run 1': stats['values'][0],
            'Run 2': stats['values'][1],
            'Run 3': stats['values'][2]
        })

per_class_df = pd.DataFrame(per_class_rows)

print("\n\n📊 PER-CLASS PERFORMANCE METRICS (3 RUNS)")
print("="*80)
# Display grouped by class
for class_name in class_names:
    class_data = per_class_df[per_class_df['Class'] == class_name]
    print(f"\n{class_name}:")
    print(class_data[['Metric', 'Mean ± Std']].to_string(index=False))
print("="*80)

# Save to CSV
per_class_df.to_csv(os.path.join(BASE_RESULT_DIR, "per_class_metrics_summary.csv"), index=False)

print("\n✅ Summary tables saved to CSV files!")

In [ ]:
# ========== GENERATE LATEX TABLE FOR PAPER ========== #
print("\n" + "="*80)
print("📄 GENERATING LATEX TABLES FOR SCIENTIFIC PAPER")
print("="*80 + "\n")

# Overall metrics LaTeX table
latex_overall = r"""\begin{table}[h]
\centering
\caption{Overall Performance Metrics of MobileNetV2 (Mean ± Std over 3 runs)}
\label{tab:mobilenetv2_overall}
\begin{tabular}{lcccc}
\hline
\textbf{Metric} & \textbf{Mean ± Std} & \textbf{Run 1} & \textbf{Run 2} & \textbf{Run 3} \\
\hline
"""

for _, row in overall_df.iterrows():
    latex_overall += f"{row['Metric']} & {row['Mean ± Std']} & {row['Run 1']:.4f} & {row['Run 2']:.4f} & {row['Run 3']:.4f} \\\\\n"

latex_overall += r"""\hline
\end{tabular}
\end{table}
"""

print("LaTeX Table - Overall Metrics:")
print(latex_overall)

# Per-class metrics LaTeX table (compact version for paper)
latex_per_class = r"""\begin{table}[h]
\centering
\caption{Per-Class Performance Metrics of MobileNetV2 (Mean ± Std over 3 runs)}
\label{tab:mobilenetv2_per_class}
\begin{tabular}{lccc}
\hline
\textbf{Class} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\hline
"""

for class_name in class_names:
    prec = per_class_stats[class_name]['precision']
    rec = per_class_stats[class_name]['recall']
    f1 = per_class_stats[class_name]['f1-score']
    latex_per_class += f"{class_name} & {prec['mean']:.4f} ± {prec['std']:.4f} & {rec['mean']:.4f} ± {rec['std']:.4f} & {f1['mean']:.4f} ± {f1['std']:.4f} \\\\\n"

latex_per_class += r"""\hline
\end{tabular}
\end{table}
"""

print("\n" + "="*80)
print("LaTeX Table - Per-Class Metrics:")
print(latex_per_class)

# Save LaTeX tables to file
with open(os.path.join(BASE_RESULT_DIR, "latex_tables.tex"), "w") as f:
    f.write("% Overall Metrics Table\n")
    f.write(latex_overall)
    f.write("\n\n% Per-Class Metrics Table\n")
    f.write(latex_per_class)

print("\n✅ LaTeX tables saved to 'latex_tables.tex'")

In [ ]:
# ========== VISUALIZATION OF RESULTS ACROSS RUNS ========== #
print("\n" + "="*80)
print("📈 CREATING VISUALIZATIONS")
print("="*80 + "\n")

# 1. Bar plot with error bars for overall metrics
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
means = [overall_stats[m]['mean'] for m in metrics]
stds = [overall_stats[m]['std'] for m in metrics]

x_pos = np.arange(len(metrics))
bars = ax.bar(x_pos, means, yerr=stds, capsize=10, alpha=0.8, color='steelblue', edgecolor='black')

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('MobileNetV2 Performance (Mean ± Std over 3 runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(metrics)
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (mean, std) in enumerate(zip(means, stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.4f}\n±{std:.4f}', 
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "overall_metrics_barplot.png"), dpi=300, bbox_inches='tight')
plt.show()

# 2. Box plot showing distribution across runs
fig, ax = plt.subplots(figsize=(10, 6))

data_for_box = [accuracies, precisions, recalls, f1_scores]
bp = ax.boxplot(data_for_box, labels=metrics, patch_artist=True, showmeans=True,
                meanprops=dict(marker='D', markerfacecolor='red', markersize=8))

for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
    patch.set_alpha(0.7)

ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Metrics Across 3 Runs', fontsize=14, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "metrics_boxplot.png"), dpi=300, bbox_inches='tight')
plt.show()

# 3. Per-class F1-Score comparison
fig, ax = plt.subplots(figsize=(12, 6))

class_f1_means = [per_class_stats[c]['f1-score']['mean'] for c in class_names]
class_f1_stds = [per_class_stats[c]['f1-score']['std'] for c in class_names]

x_pos = np.arange(len(class_names))
bars = ax.bar(x_pos, class_f1_means, yerr=class_f1_stds, capsize=5, 
              alpha=0.8, color='coral', edgecolor='black')

ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class F1-Score (Mean ± Std over 3 runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_ylim([0, 1.05])
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, (mean, std) in enumerate(zip(class_f1_means, class_f1_stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.3f}', 
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "per_class_f1score.png"), dpi=300, bbox_inches='tight')
plt.show()

print("✅ All visualizations created and saved!")

In [ ]:
# ========== GENERATE COMPREHENSIVE SUMMARY REPORT ========== #
print("\n" + "="*80)
print("📝 GENERATING COMPREHENSIVE SUMMARY REPORT")
print("="*80 + "\n")

report_lines = []
report_lines.append("="*100)
report_lines.append("MobileNetV2 - MULTI-RUN EXPERIMENT REPORT")
report_lines.append("="*100)
report_lines.append(f"\nGenerated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
report_lines.append(f"Random Seeds: {RANDOM_SEEDS}")
report_lines.append(f"Number of Runs: {len(RANDOM_SEEDS)}")

report_lines.append("\n" + "="*100)
report_lines.append("OVERALL PERFORMANCE METRICS (Mean ± Std)")
report_lines.append("="*100)
report_lines.append(f"{'Metric':<20} {'Mean ± Std':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
report_lines.append("-"*100)

for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    stats = overall_stats[metric]
    report_lines.append(f"{metric:<20} {stats['mean']:.4f} ± {stats['std']:.4f}      " + 
                       f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("="*100)

report_lines.append("\n" + "="*100)
report_lines.append("PER-CLASS PERFORMANCE METRICS (Mean ± Std)")
report_lines.append("="*100)

for class_name in class_names:
    report_lines.append(f"\nClass: {class_name}")
    report_lines.append("-"*100)
    report_lines.append(f"{'Metric':<20} {'Mean ± Std':<25} {'Run 1':<15} {'Run 2':<15} {'Run 3':<15}")
    report_lines.append("-"*100)
    
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        report_lines.append(f"{metric.capitalize():<20} {stats['mean']:.4f} ± {stats['std']:.4f}      " + 
                           f"{stats['values'][0]:<15.4f} {stats['values'][1]:<15.4f} {stats['values'][2]:<15.4f}")

report_lines.append("\n" + "="*100)
report_lines.append("INDIVIDUAL RUN DETAILS")
report_lines.append("="*100)

for run_result in all_runs_results:
    report_lines.append(f"\nRun {run_result['run']} (Seed: {run_result['seed']})")
    report_lines.append("-"*100)
    report_lines.append(f"  Accuracy:  {run_result['accuracy']:.4f}")
    report_lines.append(f"  Precision: {run_result['precision']:.4f}")
    report_lines.append(f"  Recall:    {run_result['recall']:.4f}")
    report_lines.append(f"  F1-Score:  {run_result['f1_score']:.4f}")
    report_lines.append(f"  Result Dir: {run_result['result_dir']}")

report_lines.append("\n" + "="*100)
report_lines.append("STATISTICAL SUMMARY")
report_lines.append("="*100)
report_lines.append(f"\nBest Run (by Accuracy):")
best_run_idx = np.argmax(accuracies)
report_lines.append(f"  Run {best_run_idx + 1} (Seed: {RANDOM_SEEDS[best_run_idx]})")
report_lines.append(f"  Accuracy: {accuracies[best_run_idx]:.4f}")

report_lines.append(f"\nWorst Run (by Accuracy):")
worst_run_idx = np.argmin(accuracies)
report_lines.append(f"  Run {worst_run_idx + 1} (Seed: {RANDOM_SEEDS[worst_run_idx]})")
report_lines.append(f"  Accuracy: {accuracies[worst_run_idx]:.4f}")

report_lines.append(f"\nVariability (Coefficient of Variation):")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    cv = (overall_stats[metric]['std'] / overall_stats[metric]['mean']) * 100
    report_lines.append(f"  {metric}: {cv:.2f}%")

report_lines.append("\n" + "="*100)
report_lines.append("GENERATED FILES")
report_lines.append("="*100)
report_lines.append("  ✓ overall_metrics_summary.csv - Overall metrics in CSV format")
report_lines.append("  ✓ per_class_metrics_summary.csv - Per-class metrics in CSV format")
report_lines.append("  ✓ latex_tables.tex - LaTeX tables for scientific paper")
report_lines.append("  ✓ overall_metrics_barplot.png - Bar plot with error bars")
report_lines.append("  ✓ metrics_boxplot.png - Box plot showing distribution")
report_lines.append("  ✓ per_class_f1score.png - Per-class F1-score comparison")
report_lines.append(f"  📁 run_1_seed_{RANDOM_SEEDS[0]}/ - Full results from Run 1")
report_lines.append(f"  📁 run_2_seed_{RANDOM_SEEDS[1]}/ - Full results from Run 2")
report_lines.append(f"  📁 run_3_seed_{RANDOM_SEEDS[2]}/ - Full results from Run 3")

report_lines.append("\n" + "="*100)
report_lines.append("END OF REPORT")
report_lines.append("="*100)

# Print report
report_text = "\n".join(report_lines)
print(report_text)

# Save report
with open(os.path.join(BASE_RESULT_DIR, "MULTI_RUN_SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)

print("\n✅ Comprehensive summary report saved to 'MULTI_RUN_SUMMARY_REPORT.txt'")

In [ ]:
# ========== PUBLICATION-READY MULTI-RUN METRICS ========== #
from sklearn.metrics import (classification_report, confusion_matrix,
                             balanced_accuracy_score, cohen_kappa_score,
                             matthews_corrcoef, top_k_accuracy_score,
                             roc_curve, auc, roc_auc_score)
from sklearn.preprocessing import label_binarize

print("="*80)
print("PUBLICATION-READY MULTI-RUN METRICS")
print("="*80)

class_names = all_runs_results[0]['class_names'] if 'class_names' in all_runs_results[0] else list(all_runs_results[0]['per_class_metrics'].keys())
n_classes = len(class_names)
labels = list(range(n_classes))

run_rows = []
for r in all_runs_results:
    y_t = np.asarray(r['y_true'])
    y_p = np.asarray(r['y_pred'])
    probs = np.asarray(r['pred_probs'])
    rep = classification_report(y_t, y_p, target_names=class_names,
                                output_dict=True, zero_division=0)
    row = {
        'Run': r['run'],
        'Seed': r['seed'],
        'Accuracy': np.mean(y_t == y_p),
        'Precision_weighted': rep['weighted avg']['precision'],
        'Recall_weighted': rep['weighted avg']['recall'],
        'F1_weighted': rep['weighted avg']['f1-score'],
        'Precision_macro': rep['macro avg']['precision'],
        'Recall_macro': rep['macro avg']['recall'],
        'F1_macro': rep['macro avg']['f1-score'],
        'Balanced_accuracy': balanced_accuracy_score(y_t, y_p),
        'Cohen_kappa': cohen_kappa_score(y_t, y_p),
        'MCC': matthews_corrcoef(y_t, y_p),
    }
    for k in [2, 3, 5]:
        if k < n_classes:
            row[f'Top_{k}_accuracy'] = top_k_accuracy_score(y_t, probs, k=k, labels=labels)
    run_rows.append(row)

publication_df = pd.DataFrame(run_rows)
numeric_cols = [c for c in publication_df.columns if c not in ['Run', 'Seed']]
summary_rows = []
for col in numeric_cols:
    mean = publication_df[col].mean()
    std = publication_df[col].std(ddof=1)
    summary_rows.append({
        'Metric': col,
        'Mean': mean,
        'Std': std,
        'Min': publication_df[col].min(),
        'Max': publication_df[col].max(),
        'Mean +/- Std': f"{mean:.4f} +/- {std:.4f}"
    })
publication_summary_df = pd.DataFrame(summary_rows)

publication_df.to_csv(os.path.join(BASE_RESULT_DIR, 'publication_metrics_by_run.csv'), index=False)
publication_summary_df.to_csv(os.path.join(BASE_RESULT_DIR, 'publication_metrics_summary.csv'), index=False)

print("\nPer-run publication metrics:")
print(publication_df.to_string(index=False))
print("\nMean +/- Std summary:")
print(publication_summary_df[['Metric', 'Mean +/- Std', 'Min', 'Max']].to_string(index=False))

agg_cm = np.zeros((n_classes, n_classes), dtype=int)
for r in all_runs_results:
    agg_cm += confusion_matrix(r['y_true'], r['y_pred'], labels=labels)

agg_cm_df = pd.DataFrame(agg_cm, index=class_names, columns=class_names)
agg_cm_df.to_csv(os.path.join(BASE_RESULT_DIR, 'aggregate_confusion_matrix.csv'))

row_sums = agg_cm.sum(axis=1, keepdims=True)
agg_cm_norm = np.divide(agg_cm, row_sums, out=np.zeros_like(agg_cm, dtype=float), where=row_sums != 0)
pd.DataFrame(agg_cm_norm, index=class_names, columns=class_names).to_csv(
    os.path.join(BASE_RESULT_DIR, 'aggregate_confusion_matrix_normalized.csv'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(agg_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Aggregate Confusion Matrix (3 runs)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
sns.heatmap(agg_cm_norm, annot=True, fmt='.3f', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Normalized Aggregate Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'aggregate_confusion_matrix.png'), dpi=300)
plt.close()

all_y_true = np.concatenate([np.asarray(r['y_true']) for r in all_runs_results])
all_pred_probs = np.vstack([np.asarray(r['pred_probs']) for r in all_runs_results])
y_bin = label_binarize(all_y_true, classes=labels)

auc_rows = []
fig, ax = plt.subplots(figsize=(7, 6))
for i, cname in enumerate(class_names):
    if len(np.unique(y_bin[:, i])) < 2:
        continue
    fpr, tpr, _ = roc_curve(y_bin[:, i], all_pred_probs[:, i])
    class_auc = auc(fpr, tpr)
    auc_rows.append({'Class': cname, 'AUC': class_auc})
    ax.plot(fpr, tpr, lw=2, label=f"{cname} (AUC={class_auc:.4f})")

try:
    macro_auc = roc_auc_score(all_y_true, all_pred_probs, labels=labels, multi_class='ovr', average='macro')
    weighted_auc = roc_auc_score(all_y_true, all_pred_probs, labels=labels, multi_class='ovr', average='weighted')
    auc_rows.extend([
        {'Class': 'Macro average', 'AUC': macro_auc},
        {'Class': 'Weighted average', 'AUC': weighted_auc},
    ])
except ValueError as exc:
    print(f"ROC-AUC summary skipped: {exc}")

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('MobileNetV2 ROC Curves (aggregate over 3 runs)')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves_aggregate.png'), dpi=300)
plt.close()

auc_df = pd.DataFrame(auc_rows)
auc_df.to_csv(os.path.join(BASE_RESULT_DIR, 'auc_scores.csv'), index=False)

distribution_rows = []
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(DATA_DIR, split)
    for cname in class_names:
        class_dir = os.path.join(split_dir, cname)
        n = len([f for f in os.listdir(class_dir)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
        distribution_rows.append({'Split': split, 'Class': cname, 'Samples': n})

distribution_df = pd.DataFrame(distribution_rows)
distribution_df.to_csv(os.path.join(BASE_RESULT_DIR, 'dataset_distribution.csv'), index=False)

report_lines = []
report_lines.append('MobileNetV2 - report-ready multi-run summary')
report_lines.append('='*80)
report_lines.append(f'Dataset: {DATA_DIR}')
report_lines.append(f'Random seeds: {RANDOM_SEEDS}')
report_lines.append(f'Runs: {len(all_runs_results)}')
report_lines.append(f'Classes: {", ".join(class_names)}')
report_lines.append('\nDataset distribution:')
report_lines.append(distribution_df.pivot(index='Class', columns='Split', values='Samples').to_string())
report_lines.append('\nKey metrics (mean +/- std over runs):')
for _, row in publication_summary_df.iterrows():
    report_lines.append(f"{row['Metric']}: {row['Mean +/- Std']}")
if not auc_df.empty:
    report_lines.append('\nAUC scores:')
    for _, row in auc_df.iterrows():
        report_lines.append(f"{row['Class']}: {row['AUC']:.4f}")
report_lines.append('\nGenerated files:')
report_lines.extend([
    'publication_metrics_by_run.csv',
    'publication_metrics_summary.csv',
    'aggregate_confusion_matrix.csv',
    'aggregate_confusion_matrix_normalized.csv',
    'aggregate_confusion_matrix.png',
    'roc_curves_aggregate.png',
    'auc_scores.csv',
    'dataset_distribution.csv',
])

with open(os.path.join(BASE_RESULT_DIR, 'REPORT_READY_SUMMARY.txt'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

print("\nSaved publication-ready files to:", BASE_RESULT_DIR)


In [ ]:
# ========== ZIP ALL RESULTS ========== #
import shutil

print("\n" + "="*80)
print("🗜️  CREATING COMPLETE ARCHIVE")
print("="*80 + "\n")

zip_output_path = "/kaggle/working/MobileNetV2_MultiRun_Complete"
print(f"Source: {BASE_RESULT_DIR}")
print(f"Output: {zip_output_path}.zip")

# Create zip file
shutil.make_archive(zip_output_path, 'zip', BASE_RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"\n✅ Complete multi-run report archived successfully!")
print(f"📦 Archive size: {zip_size:.2f} MB")
print(f"📍 Location: {zip_output_path}.zip")
print("\n" + "="*80)
print("🎉 ALL DONE! Ready for scientific paper submission!")
print("="*80)

In [ ]:
# ========== SELECT RUN FOR SINGLE-RUN ANALYSIS ========== #
# Change SELECTED_RUN to 1, 2, or 3 to analyse a different seed.
from types import SimpleNamespace

SELECTED_RUN = len(RANDOM_SEEDS)  # default: last run
run_data = all_runs_results[SELECTED_RUN - 1]

MODEL_FILENAME = "mobilenetv2_best.keras"
RESULT_DIR = run_data['result_dir']
y_true = np.asarray(run_data['y_true'])
y_pred = np.asarray(run_data['y_pred'])
pred_probs = np.asarray(run_data['pred_probs'])
class_names = run_data['class_names']
test_acc = run_data['accuracy']
history = SimpleNamespace(history=run_data['history'])

train_generator, val_generator, test_generator = create_generators(
    DATA_DIR, input_shape[:2], batch_size, seed=run_data['seed']
)
test_generator.reset()
model = load_model(os.path.join(RESULT_DIR, MODEL_FILENAME))

print(f"Analysing Run {SELECTED_RUN} (seed={run_data['seed']})")
print(f"  Model: MobileNetV2")
print(f"  Classes: {class_names}")
print(f"  Train / Val / Test: {train_generator.samples} / {val_generator.samples} / {test_generator.samples}")
print(f"  Accuracy: {run_data['accuracy']:.4f}")
print(f"  F1-score (weighted): {run_data['f1_score']:.4f}")
print(f"  Result dir: {RESULT_DIR}")


In [ ]:
# ========== LEARNING CURVES ========== #
plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()
plt.grid()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.grid()

plt.savefig(os.path.join(RESULT_DIR, "learning_curve.png"), dpi=300)
plt.show()

In [ ]:
# ========== LOAD BEST MODEL ========== #
model = load_model(os.path.join(RESULT_DIR, 'mobilenetv2_best.keras'))

# predict
pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

In [ ]:
# ========== CLASSIFICATION REPORT ========== #
report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)

with open(os.path.join(RESULT_DIR, "classification_report.txt"), "w") as f:
    f.write(report)

In [ ]:
# ========== CONFUSION MATRIX ========== #
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names,
            yticklabels=class_names,
            cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix ")
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "confusion_matrix.png"), dpi=300)
plt.show()

In [ ]:
# ========== LOAD BEST MODEL ========== #
model = load_model(os.path.join(RESULT_DIR, 'mobilenetv2_best.keras'))

# reset generator (quan trọng)
test_generator.reset()

# predict
pred_probs = model.predict(test_generator, verbose=1)

y_pred = np.argmax(pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

print("Total test samples:", len(y_true))

In [ ]:
test_acc = np.mean(y_pred == y_true)
print("Test Accuracy:", test_acc)

In [ ]:
# ========== PREPARE PATHS ========== #
test_dir = os.path.join(DATA_DIR, "test")
filepaths = [os.path.join(test_dir, f) for f in test_generator.filenames]

In [ ]:
# ========== FIND CONFUSION TYPES ========== #
from collections import defaultdict

confusion_dict = defaultdict(list)

for i in range(len(y_true)):
    if y_true[i] != y_pred[i]:
        key = (class_names[y_true[i]], class_names[y_pred[i]])
        confidence = pred_probs[i][y_pred[i]]
        confusion_dict[key].append((filepaths[i], confidence, i))

print("Total confusion types:", len(confusion_dict))

In [ ]:
# ========== SELECT REPRESENTATIVE IMAGES ========== #
import shutil

analysis_dir = os.path.join(RESULT_DIR, "qualitative_analysis")
os.makedirs(analysis_dir, exist_ok=True)

summary_lines = []

for (true_label, pred_label), samples in confusion_dict.items():

    # sort theo độ tự tin giảm dần
    samples_sorted = sorted(samples, key=lambda x: x[1], reverse=True)

    selected = samples_sorted[:10]  # lấy 2 ảnh

    pair_folder = os.path.join(analysis_dir, f"{true_label}_as_{pred_label}")
    os.makedirs(pair_folder, exist_ok=True)

    summary_lines.append(f"\n=== {true_label} → {pred_label} ===")

    for idx, (img_path, conf, i) in enumerate(selected):
        new_name = f"sample_{idx+1}_conf_{conf:.3f}.jpg"
        dst = os.path.join(pair_folder, new_name)
        shutil.copy(img_path, dst)

        summary_lines.append(f"{new_name} | confidence={conf:.3f}")

In [ ]:
with open(os.path.join(analysis_dir, "analysis_notes.txt"), "w") as f:
    f.write("\n".join(summary_lines))

print("Saved qualitative analysis samples")

In [ ]:
# ========== ZIP QUALITATIVE ANALYSIS ========== #
import shutil

qual_zip_base = os.path.join(RESULT_DIR, "qualitative_analysis")
qual_zip_path = shutil.make_archive(qual_zip_base, 'zip', analysis_dir)
print(f"Qualitative analysis archived: {qual_zip_path}")


# ========== MODEL ANALYSIS & REPORTS ========== #

In [ ]:
# ========== MODEL SUMMARY & PARAMETERS ========== #
model.summary()

# Count parameters
total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
non_trainable_params = total_params - trainable_params

print("\n" + "="*60)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Non-trainable params: {non_trainable_params:,}")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "model_summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))
    f.write("\n" + "="*60 + "\n")
    f.write(f"Total params: {total_params:,}\n")
    f.write(f"Trainable params: {trainable_params:,}\n")
    f.write(f"Non-trainable params: {non_trainable_params:,}\n")
    f.write("="*60 + "\n")

print("✅ Model summary saved")

In [ ]:
# ========== MODEL SIZE ========== #
import tempfile

# Save model temporarily to get size
temp_model_path = os.path.join(tempfile.gettempdir(), "temp_model.keras")
model.save(temp_model_path)
model_size_bytes = os.path.getsize(temp_model_path)
model_size_mb = model_size_bytes / (1024 * 1024)

print("="*60)
print(f"Model size: {model_size_mb:.2f} MB ({model_size_bytes:,} bytes)")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "model_size.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write(f"Model size: {model_size_mb:.2f} MB ({model_size_bytes:,} bytes)\n")
    f.write("="*60 + "\n")

# Clean up
os.remove(temp_model_path)
print("✅ Model size saved")

In [ ]:
# ========== INFERENCE SPEED ========== #
import time

# Warm-up: run a few predictions to initialize
print("Warming up...")
test_generator.reset()
warmup_batch = next(test_generator)[0][:5]  # 5 images
for _ in range(3):
    _ = model.predict(warmup_batch, verbose=0)

# Measure inference time on multiple batches
print("\nMeasuring inference speed...")
test_generator.reset()
num_test_batches = 10
total_images = 0
total_time = 0

for i in range(num_test_batches):
    batch_x, _ = next(test_generator)
    batch_size_actual = len(batch_x)
    
    start_time = time.time()
    _ = model.predict(batch_x, verbose=0)
    end_time = time.time()
    
    total_images += batch_size_actual
    total_time += (end_time - start_time)

# Calculate metrics
avg_time_per_image = (total_time / total_images) * 1000  # ms
fps = total_images / total_time

print("\n" + "="*60)
print("Inference Speed:")
print(f"  FPS: {fps:.2f}")
print(f"  ms/image: {avg_time_per_image:.2f}")
print(f"  Total images tested: {total_images}")
print(f"  Total time: {total_time:.3f}s")
print("="*60)

# Save to file
with open(os.path.join(RESULT_DIR, "inference_speed.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("Inference Speed:\n")
    f.write(f"  FPS: {fps:.2f}\n")
    f.write(f"  ms/image: {avg_time_per_image:.2f}\n")
    f.write(f"  Total images tested: {total_images}\n")
    f.write(f"  Total time: {total_time:.3f}s\n")
    f.write("="*60 + "\n")

print("✅ Inference speed saved")

In [ ]:
# ========== TOP-K ACCURACY ========== #
from sklearn.metrics import top_k_accuracy_score

num_classes = len(class_names)
labels = list(range(num_classes))
top_k_results = {}
top_k_notes = []

top_1_acc = top_k_accuracy_score(y_true, pred_probs, k=1, labels=labels)
top_k_results[1] = top_1_acc

# Top-k is only informative when k < number of classes.
# With 3 garlic classes, Top-3 and Top-5 are always 1.0 and should not be reported.
for k in [2, 3, 5]:
    if k < num_classes:
        top_k_results[k] = top_k_accuracy_score(y_true, pred_probs, k=k, labels=labels)
    else:
        top_k_notes.append(f"Top-{k} skipped because k >= number of classes ({num_classes}).")

top_2_acc = top_k_results.get(2, np.nan)
top_3_acc = top_k_results.get(3, np.nan)
top_5_acc = top_k_results.get(5, np.nan)

topk_report_lines = [
    f"Top-{k} Accuracy: {v:.4f} ({v*100:.2f}%)"
    for k, v in sorted(top_k_results.items())
]
topk_report_lines.extend(top_k_notes)

print("\n" + "="*60)
print("Top-K Accuracy:")
for line in topk_report_lines:
    print("  " + line)
print("="*60)

with open(os.path.join(RESULT_DIR, "topk_accuracy.txt"), "w", encoding="utf-8") as f:
    f.write("="*60 + "\n")
    f.write("Top-K Accuracy:\n")
    for line in topk_report_lines:
        f.write(f"  {line}\n")
    f.write("="*60 + "\n")

print("Top-K accuracy saved")


In [ ]:
# ========== PER-CLASS ACCURACY ========== #
from sklearn.metrics import classification_report

# Get per-class metrics
report_dict = classification_report(y_true, y_pred, target_names=class_names, 
                                   output_dict=True, digits=4)

# Create per-class accuracy dataframe
per_class_df = pd.DataFrame({
    'Class': class_names,
    'Precision': [report_dict[c]['precision'] for c in class_names],
    'Recall': [report_dict[c]['recall'] for c in class_names],
    'F1-Score': [report_dict[c]['f1-score'] for c in class_names],
    'Support': [report_dict[c]['support'] for c in class_names]
})

print("\n" + "="*60)
print("Per-Class Metrics:")
print(per_class_df.to_string(index=False))
print("="*60)

# Save to CSV
per_class_df.to_csv(os.path.join(RESULT_DIR, "per_class_metrics.csv"), index=False)
print("✅ Per-class metrics saved")

In [ ]:
# ========== PREDICTIONS CSV ========== #
# Create detailed predictions dataframe
predictions_df = pd.DataFrame({
    'filename': test_generator.filenames,
    'true_label': [class_names[i] for i in y_true],
    'predicted_label': [class_names[i] for i in y_pred],
    'correct': y_true == y_pred,
    'confidence': [pred_probs[i][y_pred[i]] for i in range(len(y_pred))]
})

# Add top-3 predictions for each image
for k in range(min(3, len(class_names))):
    top_k_indices = np.argsort(pred_probs, axis=1)[:, -(k+1)]
    predictions_df[f'top_{k+1}_class'] = [class_names[i] for i in top_k_indices]
    predictions_df[f'top_{k+1}_prob'] = [pred_probs[i][top_k_indices[i]] for i in range(len(pred_probs))]

# Save to CSV
predictions_df.to_csv(os.path.join(RESULT_DIR, "predictions_detail.csv"), index=False)

print(f"✅ Predictions CSV saved ({len(predictions_df)} samples)")
print(f"   Correct predictions: {predictions_df['correct'].sum()}")
print(f"   Incorrect predictions: {(~predictions_df['correct']).sum()}")

In [ ]:
# ========== COMPREHENSIVE SUMMARY REPORT ========== #
summary_report = []
summary_report.append("="*80)
summary_report.append("MobileNetV2 - COMPREHENSIVE EVALUATION REPORT")
summary_report.append("="*80)
summary_report.append(f"\nDataset: {DATA_DIR}")
summary_report.append(f"Result Directory: {RESULT_DIR}")
summary_report.append(f"Training Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")

summary_report.append("\n" + "-"*80)
summary_report.append("MODEL CONFIGURATION")
summary_report.append("-"*80)
summary_report.append(f"Architecture: MobileNetV2")
summary_report.append(f"Input Shape: {input_shape}")
summary_report.append(f"Number of Classes: {len(class_names)}")
summary_report.append(f"Classes: {', '.join(class_names)}")
summary_report.append(f"\nTotal Parameters: {total_params:,}")
summary_report.append(f"Trainable Parameters: {trainable_params:,}")
summary_report.append(f"Non-trainable Parameters: {non_trainable_params:,}")
summary_report.append(f"Model Size: {model_size_mb:.2f} MB")

summary_report.append("\n" + "-"*80)
summary_report.append("DATASET STATISTICS")
summary_report.append("-"*80)
summary_report.append(f"Training Samples: {train_generator.samples}")
summary_report.append(f"Validation Samples: {val_generator.samples}")
summary_report.append(f"Test Samples: {test_generator.samples}")

summary_report.append("\n" + "-"*80)
summary_report.append("TRAINING CONFIGURATION")
summary_report.append("-"*80)
summary_report.append(f"Batch Size: {batch_size}")
summary_report.append(f"Total Epochs: {len(history.history['loss'])}")
summary_report.append(f"Initial Learning Rate: 1e-5")
summary_report.append(f"Optimizer: Adam with ExponentialDecay")
summary_report.append(f"Loss Function: CategoricalCrossentropy (label_smoothing=0.15)")
summary_report.append(f"Class Weights: Balanced")

summary_report.append("\n" + "-"*80)
summary_report.append("PERFORMANCE METRICS")
summary_report.append("-"*80)
summary_report.append(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
if 'topk_report_lines' in globals():
    summary_report.extend(topk_report_lines)
else:
    summary_report.append(f"Top-1 Accuracy: {top_1_acc:.4f} ({top_1_acc*100:.2f}%)")

summary_report.append("\n" + "-"*80)
summary_report.append("INFERENCE SPEED")
summary_report.append("-"*80)
summary_report.append(f"FPS: {fps:.2f}")
summary_report.append(f"ms/image: {avg_time_per_image:.2f}")

summary_report.append("\n" + "-"*80)
summary_report.append("BEST TRAINING EPOCH METRICS")
summary_report.append("-"*80)
best_val_loss_idx = np.argmin(history.history['val_loss'])
summary_report.append(f"Best Epoch: {best_val_loss_idx + 1}")
summary_report.append(f"  Train Loss: {history.history['loss'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Train Accuracy: {history.history['accuracy'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Val Loss: {history.history['val_loss'][best_val_loss_idx]:.4f}")
summary_report.append(f"  Val Accuracy: {history.history['val_accuracy'][best_val_loss_idx]:.4f}")

summary_report.append("\n" + "="*80)
summary_report.append("END OF REPORT")
summary_report.append("="*80)

# Print and save
summary_text = "\n".join(summary_report)
print(summary_text)

with open(os.path.join(RESULT_DIR, "SUMMARY_REPORT.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\n✅ Comprehensive summary report saved")

In [ ]:
# ========== LIST ALL REPORT FILES ========== #
import glob

print("\n" + "="*80)
print("GENERATED REPORT FILES:")
print("="*80)

all_files = glob.glob(os.path.join(RESULT_DIR, "*"))
for file_path in sorted(all_files):
    if os.path.isfile(file_path):
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path)
        if file_size < 1024:
            size_str = f"{file_size} B"
        elif file_size < 1024*1024:
            size_str = f"{file_size/1024:.2f} KB"
        else:
            size_str = f"{file_size/(1024*1024):.2f} MB"
        print(f"  ✓ {file_name:40s} ({size_str})")
    elif os.path.isdir(file_path):
        dir_name = os.path.basename(file_path)
        num_files = len([f for f in glob.glob(os.path.join(file_path, "**/*"), recursive=True) if os.path.isfile(f)])
        print(f"  📁 {dir_name:40s} ({num_files} files)")

print("="*80)

In [ ]:
# ========== ZIP ALL REPORTS ========== #
import shutil

zip_output_path = "/kaggle/working/MobileNetV2_Complete_Report"
print(f"\n🗜️  Creating complete report archive...")
print(f"Source: {RESULT_DIR}")
print(f"Output: {zip_output_path}.zip")

# Create zip file
shutil.make_archive(zip_output_path, 'zip', RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"\n✅ Complete report archived successfully!")
print(f"📦 Archive size: {zip_size:.2f} MB")
print(f"📍 Location: {zip_output_path}.zip")
print("\n" + "="*80)
print("ARCHIVE CONTENTS:")
print("  ✓ SUMMARY_REPORT.txt - Comprehensive evaluation summary")
print("  ✓ model_summary.txt - Model architecture & parameters")
print("  ✓ model_size.txt - Model file size")
print("  ✓ inference_speed.txt - FPS & latency metrics")
print("  ✓ topk_accuracy.txt - Top-1, Top-3, Top-5 accuracy")
print("  ✓ classification_report.txt - Precision, Recall, F1 per class")
print("  ✓ per_class_metrics.csv - Detailed class metrics")
print("  ✓ predictions_detail.csv - All predictions with confidence")
print("  ✓ training_log.csv - Training history")
print("  ✓ learning_curve.png - Training visualization")
print("  ✓ confusion_matrix.png - Confusion matrix heatmap")
print("  ✓ mobilenetv2_best.keras - Best model weights")
print("  📁 qualitative_analysis/ - Misclassified samples analysis")
print("="*80)